# Proyecto BI - Uber NYC (2009-2015)
## Fase 3: Preparacion y Modelado de Datos (ETL)
Curso: Analisis de Datos - ITM 2026-1

Este notebook ejecuta el proceso ETL para dejar el dataset listo para analisis en Power BI o Tableau.

Resumen de resultados de esta fase (valores reales):
- Dataset procesado: uber.csv
- Registros iniciales: 200,000
- Registros finales: 191,133
- Retencion total: 95.57%
- Rango temporal final: 2009 a 2015

In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

def print_section(title: str):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

### 3.1 Extraccion y exploracion inicial

Se carga el dataset uber.csv con registros de viajes de Uber en Nueva York.

Hallazgos principales de la exploracion inicial:
- Filas iniciales: 200,000
- Columnas utiles: 8
- Registros duplicados detectados: 0
- Filas con al menos un nulo: 1

Se aplica inspeccion con info(), head() y describe() para validar tipos de datos y distribuciones base antes de limpiar.

In [9]:
print_section("3.1 EXTRACCION Y EXPLORACION INICIAL")

df = pd.read_csv("uber.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

required_columns = [
    "fare_amount", "pickup_datetime",
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "passenger_count"
 ]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

print("Fuente: uber.csv")
print(f"Filas originales: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print("\nInfo general:")
df.info()
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")
print(f"\nPrimeras filas:\n{df.head(3)}")
print(f"\nEstadisticas descriptivas:\n{df.describe(include='all').transpose()}")


3.1 EXTRACCION Y EXPLORACION INICIAL
Fuente: uber.csv
Filas originales: 200,000
Columnas: 8

Info general:
<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   key                200000 non-null  str    
 1   fare_amount        200000 non-null  float64
 2   pickup_datetime    200000 non-null  str    
 3   pickup_longitude   200000 non-null  float64
 4   pickup_latitude    200000 non-null  float64
 5   dropoff_longitude  199999 non-null  float64
 6   dropoff_latitude   199999 non-null  float64
 7   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(1), str(2)
memory usage: 12.2 MB

Valores nulos por columna:
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64

Primeras filas:
  

### 3.2 Limpieza de datos

Para garantizar calidad del dataset se aplican los siguientes filtros (con resultados reales):
- Duplicados eliminados: 0
- Registros con nulos en columnas criticas: 1
- Tarifas invalidas (fare_amount <= 0 o > 200): 29
- Coordenadas fuera del rango de NYC o invalidas: 4,464
- passenger_count fuera del rango 1 a 6: 686

Total eliminado en limpieza base: 5,180 registros.

In [10]:
print_section("3.2 LIMPIEZA DE DATOS")

filas_antes = len(df)

df.drop_duplicates(inplace=True)
print(f"Duplicados eliminados: {filas_antes - len(df):,}")

numeric_critical_columns = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
 ]
for col in numeric_critical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

critical_columns = ["pickup_datetime"] + numeric_critical_columns
df.dropna(subset=critical_columns, inplace=True)
print(f"Filas tras eliminar nulos: {len(df):,}")

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], utc=True, errors="coerce")
df.dropna(subset=["pickup_datetime"], inplace=True)

mask_fare = (df["fare_amount"] > 0) & (df["fare_amount"] <= 200)
eliminados_fare = (~mask_fare).sum()
df = df[mask_fare]
print(f"Filas eliminadas por tarifa invalida (<=0 o >200): {eliminados_fare:,}")

def coordenadas_validas_nyc(lat, lon):
    lat_min, lat_max = 40.4774, 40.9176
    lon_min, lon_max = -74.2591, -73.7004
    return (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)

mask_pickup = coordenadas_validas_nyc(df["pickup_latitude"], df["pickup_longitude"])
mask_dropoff = coordenadas_validas_nyc(df["dropoff_latitude"], df["dropoff_longitude"])
eliminados_coords = (~(mask_pickup & mask_dropoff)).sum()
df = df[mask_pickup & mask_dropoff].copy()
print(f"Filas eliminadas por coordenadas invalidas: {eliminados_coords:,}")

mask_pax = (df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)
eliminados_pax = (~mask_pax).sum()
df = df[mask_pax].copy()
print(f"Filas eliminadas por passenger_count invalido (fuera de 1 a 6): {eliminados_pax:,}")

print(f"\nFilas finales tras limpieza: {len(df):,}")
print(f"Porcentaje de datos conservados tras limpieza: {len(df) / filas_antes * 100:.2f}%")


3.2 LIMPIEZA DE DATOS
Duplicados eliminados: 0
Filas tras eliminar nulos: 199,999
Filas eliminadas por tarifa invalida (<=0 o >200): 29
Filas eliminadas por coordenadas invalidas: 4,464
Filas eliminadas por passenger_count invalido (fuera de 1 a 6): 686

Filas finales tras limpieza: 194,820
Porcentaje de datos conservados tras limpieza: 97.41%


### 3.3 Transformacion de datos

A partir de pickup_datetime se crean las columnas year, month, day_of_week y hour.
Tambien se calcula distance_km con la formula de Haversine y se clasifica time_of_day en:
- Madrugada (0-5)
- Manana (6-11)
- Tarde (12-17)
- Noche (18-23)

Complemento de calidad aplicado:
- Eliminados por distance_km < 0.1: 3,099
- Eliminados por fare_per_km > 30: 588

Indicadores posteriores a la transformacion:
- Distancia promedio: 3.37 km
- Tarifa promedio: 11.26 USD
- Pasajeros promedio: 1.69
- Distribucion por franja horaria: Noche 34.40%, Tarde 28.65%, Manana 24.11%, Madrugada 12.84%

In [ ]:
print_section("3.3 TRANSFORMACION DE DATOS")

df["year"] = df["pickup_datetime"].dt.year
df["month"] = df["pickup_datetime"].dt.month
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek
df["hour"] = df["pickup_datetime"].dt.hour
df["month_name"] = df["pickup_datetime"].dt.strftime("%B")
df["day_name"] = df["pickup_datetime"].dt.strftime("%A")

def franja_horaria(h):
    if 0 <= h < 6:
        return "Madrugada"
    if 6 <= h < 12:
        return "Manana"
    if 12 <= h < 18:
        return "Tarde"
    return "Noche"

df["time_of_day"] = df["hour"].apply(franja_horaria)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df["distance_km"] = haversine(
    df["pickup_latitude"].values,
    df["pickup_longitude"].values,
    df["dropoff_latitude"].values,
    df["dropoff_longitude"].values,
)

mask_dist = df["distance_km"] >= 0.1
eliminados_dist = (~mask_dist).sum()
df = df[mask_dist].copy()
print(f"Filas eliminadas por distancia < 0.1 km: {eliminados_dist:,}")

df["fare_per_km"] = df["fare_amount"] / df["distance_km"]
mask_fpkm = df["fare_per_km"] <= 30
eliminados_fpkm = (~mask_fpkm).sum()
df = df[mask_fpkm].copy()
print(f"Filas eliminadas por fare_per_km > 30: {eliminados_fpkm:,}")

print("\nVariables temporales y distancia creadas exitosamente.")

### 3.4 Seleccion de variables - Dimensiones y metricas

Se seleccionan las variables clave para analisis en BI.

| Tipo | Variable | Descripcion |
|------|----------|-------------|
| Dimension | year | Ano del viaje |
| Dimension | month | Mes del viaje |
| Dimension | day_of_week | Dia de la semana |
| Dimension | hour | Hora del viaje |
| Dimension | time_of_day | Franja horaria categorizada |
| Metrica | fare_amount | Tarifa cobrada en USD |
| Metrica | distance_km | Distancia recorrida en kilometros |
| Metrica | passenger_count | Numero de pasajeros por viaje |

Nota: fare_per_km se mantiene como variable tecnica de control de calidad, no como metrica principal de consumo.

In [ ]:
print_section("3.4 SELECCION DE VARIABLES PARA EL DASHBOARD")

DIMENSIONES = {
    "year": "Ano del viaje",
    "month": "Mes del viaje (1-12)",
    "day_of_week": "Dia de la semana (0=Lunes ... 6=Domingo)",
    "hour": "Hora del dia (0-23)",
    "time_of_day": "Franja horaria",
}

METRICAS = {
    "fare_amount": "Tarifa cobrada en USD",
    "distance_km": "Distancia del viaje en km",
    "passenger_count": "Numero de pasajeros por viaje",
}

print("DIMENSIONES:")
for col, desc in DIMENSIONES.items():
    print(f" - {col:<18} : {desc}")

print("\nMETRICAS:")
for col, desc in METRICAS.items():
    print(f" - {col:<18} : {desc}")

print("\nVariable tecnica de control:")
print(" - fare_per_km         : Tarifa por kilometro (solo para control de calidad)")


SELECCION DE VARIABLES PARA EL DASHBOARD
DIMENSIONES:
 - year               : Ano del viaje (temporal)
 - month              : Mes del viaje (1-12)
 - month_name         : Nombre del mes
 - day_of_week        : Dia de la semana (0=Lunes ... 6=Domingo)
 - day_name           : Nombre del dia
 - hour               : Hora del dia (0-23)
 - time_of_day        : Franja horaria

METRICAS:
 - fare_amount        : Tarifa cobrada en USD
 - distance_km        : Distancia del viaje en km
 - passenger_count    : Numero de pasajeros por viaje
 - fare_per_km        : Tarifa por kilometro

COORDENADAS:
 - pickup_latitude    : Latitud de recogida
 - pickup_longitude   : Longitud de recogida
 - dropoff_latitude   : Latitud de destino
 - dropoff_longitude  : Longitud de destino


### 3.5 Exportacion del dataset limpio

Se construye el dataframe final con 8 variables (5 dimensiones + 3 metricas) y se exporta para consumo en visualizacion.

Resultado final del ETL (valores reales):
- Shape final: 191,133 filas x 8 columnas
- Retencion total respecto al origen: 95.57%

El archivo de salida se guarda como uber_clean.csv para conservar uber.csv como fuente original.

In [ ]:
print_section("3.5 DATAFRAME FINAL Y EXPORT")

columnas_finales = list(DIMENSIONES.keys()) + list(METRICAS.keys())
df_final = df[columnas_finales].reset_index(drop=True)

print("Head (5 filas):")
print(df_final.head().to_string(index=False))

print("\nShape del dataset limpio:")
print(f"{df_final.shape[0]:,} filas x {df_final.shape[1]} columnas")

print("\nEstadisticas descriptivas (metricas):")
print(df_final[list(METRICAS.keys())].describe().round(2).to_string())

df_final.to_csv("uber_clean.csv", index=False)
print("\nDataset limpio guardado como 'uber_clean.csv'")


DATAFRAME FINAL PARA ANALISIS
Head (5 filas):
 year  month month_name  day_of_week day_name  hour    time_of_day  fare_amount  distance_km  passenger_count  fare_per_km  pickup_latitude  pickup_longitude  dropoff_latitude  dropoff_longitude
 2015      5        May            3 Thursday    19 Noche (18-23h)          7.5     1.683323                1     4.455474        40.738354        -73.999817         40.723217         -73.999512
 2009      7       July            4   Friday    20 Noche (18-23h)          7.7     2.457590                1     3.133151        40.728225        -73.994355         40.750325         -73.994710
 2009      8     August            0   Monday    21 Noche (18-23h)         12.9     5.036377                1     2.561365        40.740770        -74.005043         40.772647         -73.962565
 2009      6       June            4   Friday     8 Manana (6-11h)          5.3     1.661683                3     3.189536        40.790844        -73.976124         40.8033